In [ ]:
!pip install diffusers transformers accelerate controlnet-aux
!pip install git+https://github.com/openai/CLIP.git
!pip install segment-anything

In [ ]:
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers import UniPCMultistepScheduler
from transformers import pipeline as hf_pipeline
from PIL import Image
import requests
from io import BytesIO
import torch
import numpy as np

print("All imports done!")

In [ ]:
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config
)
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

print("Pipeline loaded!")

In [ ]:
depth_estimator = hf_pipeline(
    "depth-estimation",
    model="LiheYoung/depth-anything-base-hf"
)

print("Depth estimator loaded!")

In [ ]:
!pip install diffusers transformers accelerate controlnet-aux xformers

In [ ]:
url = "https://images.unsplash.com/photo-1555041469-a586c61ea9bc?w=512"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
image = Image.open(BytesIO(response.content)).convert("RGB")
image = image.resize((512, 512))

print("Image loaded!")
image

In [ ]:
depth_result = depth_estimator(image)
depth_map = depth_result["depth"].resize((512, 512))

print("Depth map ready!")
depth_map

In [ ]:
result = pipe(
    prompt="a modern scandinavian living room, \
            warm lighting, minimalist furniture, \
            hardwood floors, photorealistic, \
            interior design magazine, 8k",
    
    negative_prompt="ugly, blurry, bad proportions, \
                     unrealistic, cartoon, drawing, \
                     dark, gloomy",
    
    image=depth_map,
    num_inference_steps=20,
    guidance_scale=7.5
).images[0]

print("Generation complete!")
result

In [ ]:
styles = [
    "a modern scandinavian living room, warm lighting, minimalist, photorealistic",
    "a luxurious royal living room, gold accents, velvet furniture, photorealistic",
    "an industrial loft, exposed brick walls, metal fixtures, photorealistic",
    "a bohemian cozy room, warm earth tones, plants everywhere, photorealistic"
]

results = []

for style in styles:
    output = pipe(
        prompt=style,
        negative_prompt="ugly, blurry, unrealistic, cartoon",
        image=depth_map,
        num_inference_steps=20,
        guidance_scale=7.5
    ).images[0]
    results.append(output)
    print(f"Generated: {style[:40]}...")

print("All styles generated!")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
style_names = ["Scandinavian", "Royal Luxury", "Industrial", "Bohemian"]

for idx, (ax, result, name) in enumerate(zip(axes.flatten(), results, style_names)):
    ax.imshow(result)
    ax.set_title(name, fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.savefig("style_comparison.png")
plt.show()
print("Saved!")